# MERGE Bimodal Complete (BC) — MERT + RoBERTa, Audio / Text / Bimodal

Notebook ini membandingkan enam arsitektur fusion pada dataset **MERGE Bimodal Complete (BC)**.
Semua eksperimen memakai **MERT-v1-95M** sebagai audio encoder dan **RoBERTa** sebagai text encoder.
Setiap arsitektur dijalankan dalam tiga konfigurasi modalitas:

- `audio` — hanya audio
- `text` — hanya lirik
- `bimodal` — audio + lirik

Dua skenario split disediakan: `70/15/15` dan `40/30/30`. Split resmi MERGE dicoba terlebih
dahulu; bila memecah duplikat lagu, notebook memakai split *stratified group-aware* dari
song ID. Semua model memakai split yang sama pada repeat yang sama.

**Status eksekusi:** full run pada 2.172 lagu sudah dijalankan pada 2026-11-18
dengan `N_TRIALS=5`, `MAX_EPOCHS=30`, `PATIENCE=15`, dan `REPEATS=3`.
Hasil lengkap ada di `experiments_mert/`; smoke test cepat ada di `experiments_mert_quick/`
dan tidak boleh dipakai sebagai hasil penelitian.

**Perubahan dari notebook asli:** `CrossModalMTL`, genre MTL, dan pipeline mel/GloVe diganti
dengan MERT + RoBERTa. Infrastruktur path, QC manifest, split, metrik, early stopping, cache,
dan penanganan GPU/CPU dipertahankan. Backup notebook asli: `merge_attention_mtl_backup.ipynb`.

## Ringkasan perubahan dari notebook asli

- Dataset tetap MERGE Bimodal Complete; QC, exclusion log, dan split resmi dipertahankan.
- `CrossModalMTL`, genre MTL, dan pipeline mel/GloVe diganti dengan MERT + RoBERTa.
- Setiap arsitektur dijalankan untuk `audio`, `text`, dan `bimodal`.
- Split resmi 70/15/15 dan 40/30/30 dipakai; repeat tambahan dibuat group-aware.
- Ditambahkan cache MERT/RoBERTa, Optuna, early stopping patience 15, dan evaluasi lengkap.
- Backup notebook asli tersedia di `merge_attention_mtl_backup.ipynb`.

## 1. Import library dan konfigurasi

## Hasil evaluasi lengkap

Full run 2.172 lagu sudah dijalankan. Laporan lengkap tersedia di:

- `LAPORAN_EVALUASI_MERGE_MERT.md` (tabel evaluasi, per-class metrics, confusion matrix, fusion/occlusion, gate/alpha, hyperparameter, rekomendasi)
- `experiments_mert/model_comparison_tvt_70_15_15.csv`
- `experiments_mert/model_comparison_tvt_40_30_30.csv`
- `experiments_mert/evaluation_results.csv` (108 final run)
- `experiments_mert/confusion_matrix.csv`
- `experiments_mert/classification_report.csv`

Tabel dan grafik di bagian 20-23 notebook dapat diregenerasi dari CSV tersebut.


In [ ]:
import os
import sys
import json
import re
import time
import math
import random
import hashlib
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import matplotlib.pyplot as plt
import optuna

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def find_project_root(start=None):
    """Cari root yang berisi metadata MERGE, baik layout data/ maupun folder zip."""
    env = os.environ.get("MERGE_ROOT")
    candidates = []
    if env:
        candidates.append(Path(env).expanduser())
    p = Path(start or Path.cwd()).resolve()
    candidates.extend([p, *p.parents])
    markers = [
        Path("MERGE_Bimodal_Complete") / "merge_bimodal_complete_metadata.csv",
        Path("data") / "merge_bimodal_complete_metadata.csv",
        Path("merge_bimodal_complete_metadata.csv"),
    ]
    for cand in candidates:
        for marker in markers:
            if (cand / marker).exists():
                return cand
    return p


ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


DATA_DIR = Path(os.environ.get("MERGE_DATA_DIR", ROOT / "MERGE_Bimodal_Complete"))
if not DATA_DIR.exists():
    DATA_DIR = ROOT / "data"

META_CSV = first_existing([
    DATA_DIR / "merge_bimodal_complete_metadata.csv",
    DATA_DIR / "metadata.csv",
    ROOT / "merge_bimodal_complete_metadata.csv",
])
AUDIO_ROOTS = [d for d in [
    DATA_DIR / "audio",
    DATA_DIR / "MERGE_Bimodal_Complete" / "audio",
    ROOT / "audio",
] if d.exists()]
LYRIC_ROOTS = [d for d in [
    DATA_DIR / "lyrics",
    DATA_DIR / "MERGE_Bimodal_Complete" / "lyrics",
    ROOT / "lyrics",
] if d.exists()]
SPLIT_ROOTS = [d for d in [
    DATA_DIR / "tvt_dataframes",
    DATA_DIR / "MERGE_Bimodal_Complete" / "tvt_dataframes",
    ROOT / "tvt_dataframes",
] if d.exists()]

_QUICK_ENV = os.environ.get("MERGE_QUICK", "0") == "1"
_SUFFIX = "_quick" if _QUICK_ENV else ""
PROC = ROOT / f"processed_mert{_SUFFIX}"
FEAT = PROC / "features"
EXP = ROOT / f"experiments_mert{_SUFFIX}"
CKPT = ROOT / f"checkpoints_mert{_SUFFIX}"
OPTUNA_DIR = ROOT / f"optuna_mert{_SUFFIX}"
FIG = EXP / "figures"
for d in [PROC, FEAT, EXP, CKPT, OPTUNA_DIR, FIG]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ["Q1", "Q2", "Q3", "Q4"]
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
MODALITIES = ["audio", "text", "bimodal"]

# Encoder dan batas input
MERT_ID = "m-a-p/MERT-v1-95M"
MERT_SR = 24_000
AUDIO_DIM = 768
ROBERTA_ID = "roberta-large"
ROBERTA_FALLBACK_ID = "roberta-base"
TEXT_DIM = 1024
MAX_LYRICS_TOKENS = 512
TEXT_SEQ_LEN = 128
AUDIO_SEQ_LEN = 128
MAX_AUDIO_SEC = 30.0
QUICK_AUDIO_SEC = 5.0
MIN_AUDIO_SEC = 29.12

# Eksperimen
SCENARIOS = {
    "tvt_70_15_15": (0.70, 0.15, 0.15),
    "tvt_40_30_30": (0.40, 0.30, 0.30),
}
REPEATS = int(os.environ.get("MERGE_REPEATS", "3"))
N_TRIALS = int(os.environ.get("MERGE_TRIALS", "10"))
N_TRIALS_UNIMODAL = int(os.environ.get("MERGE_TRIALS_UNIMODAL", "5"))
MAX_EPOCHS = int(os.environ.get("MERGE_EPOCHS", "50"))
PATIENCE = int(os.environ.get("MERGE_PATIENCE", "15"))
FINAL_SEEDS = [42, 43, 44]
TUNE_SEED = 42
RESUME = True
FORCE_REEXTRACT = False
RUN_MERT_MULTILAYER = False
TUNE_UNIMODAL = False

# Mode cepat untuk smoke test; aktifkan lewat environment variable.
QUICK_RUN = _QUICK_ENV
QUICK_N = int(os.environ.get("MERGE_QUICK_N", "80"))
STRICT_CLASS_COVERAGE = True
if QUICK_RUN:
    REPEATS = 1
    N_TRIALS = 2
    N_TRIALS_UNIMODAL = 1
    MAX_EPOCHS = 10
    FINAL_SEEDS = [42]
    MAX_AUDIO_SEC = QUICK_AUDIO_SEC
    MIN_AUDIO_SEC = 0.0
if REPEATS > 1 and not QUICK_RUN:
    # Stabilitas utama berasal dari repeated split; satu seed per repeat sudah cukup.
    FINAL_SEEDS = [42]

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    ROBERTA_BATCH = 2 if _vram_gb < 6 else 8
    if _vram_gb < 6:
        BATCH_CHOICES_BIMODAL = [16, 32, 64]
        BATCH_CHOICES_AUDIO = [32, 64, 128]
    else:
        BATCH_CHOICES_BIMODAL = [16, 32, 64, 128]
        BATCH_CHOICES_AUDIO = [32, 64, 128, 256]
else:
    ROBERTA_BATCH = 4
    BATCH_CHOICES_BIMODAL = [16, 32, 64, 128]
    BATCH_CHOICES_AUDIO = [32, 64, 128, 256]
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def slugify(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")


set_seed(SEED)
print("Project root :", ROOT)
print("Data dir     :", DATA_DIR)
print("Metadata     :", META_CSV)
print("Split roots  :", SPLIT_ROOTS)
print("Device       :", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU          : {props.name} | VRAM {props.total_memory / 1024**3:.1f} GB")
else:
    print("GPU tidak tersedia; notebook memakai CPU. Ekstraksi MERT/RoBERTa akan lambat.")
print("Quick run    :", QUICK_RUN)

import platform
import importlib.metadata as _md


def _pkg_version(name):
    try:
        return _md.version(name)
    except Exception:
        return None


env_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers": _pkg_version("transformers"),
    "optuna": _pkg_version("optuna"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": _pkg_version("scikit-learn"),
    "soundfile": _pkg_version("soundfile"),
    "librosa": _pkg_version("librosa"),
    "mert_checkpoint": MERT_ID,
    "roberta_checkpoint": ROBERTA_ID,
    "seed": SEED,
    "scenarios": list(SCENARIOS),
    "repeats": REPEATS,
    "n_trials": N_TRIALS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "quick_run": QUICK_RUN,
}
(EXP / "environment.json").write_text(json.dumps(env_info, indent=2))
(EXP / "config.json").write_text(json.dumps({
    "data_dir": str(DATA_DIR), "metadata": str(META_CSV),
    "class_order": CLASS_ORDER, "max_audio_sec": MAX_AUDIO_SEC,
    "max_lyrics_tokens": MAX_LYRICS_TOKENS,
    "audio_seq_len": AUDIO_SEQ_LEN, "text_seq_len": TEXT_SEQ_LEN,
    "final_seeds": FINAL_SEEDS,
}, indent=2))

## 2. Load dataset MERGE Bimodal Complete

In [ ]:
if META_CSV is None:
    raise FileNotFoundError(
        "Metadata MERGE tidak ditemukan. Set MERGE_ROOT atau MERGE_DATA_DIR, "
        "atau letakkan merge_bimodal_complete_metadata.csv di folder proyek."
    )

meta = pd.read_csv(META_CSV, dtype=str)
meta.columns = [c.strip() for c in meta.columns]
print("Jumlah baris metadata:", len(meta))
print("Kolom:", list(meta.columns))


def pick_col(frame, names):
    lower = {str(c).strip().lower(): c for c in frame.columns}
    for name in names:
        if name.lower() in lower:
            return lower[name.lower()]
    return None


AUDIO_COL = pick_col(meta, ["Audio_Song", "Song", "audio_song"])
LYRIC_COL = pick_col(meta, ["Lyric_Song", "Lyrics_Song", "lyric_song"])
LABEL_COL = pick_col(meta, ["Quadrant", "Label", "Emotion"])
ARTIST_COL = pick_col(meta, ["Artist"])
TITLE_COL = pick_col(meta, ["Title"])

if AUDIO_COL is None or LABEL_COL is None:
    raise ValueError(f"Kolom audio/label tidak ditemukan. Kolom tersedia: {list(meta.columns)}")
if LYRIC_COL is None:
    LYRIC_COL = AUDIO_COL
    print("Kolom Lyric_Song tidak ada; memakai Song yang sama untuk audio dan lirik.")


def normalize_label(value):
    s = str(value).strip().upper()
    if s in {"Q1", "1", "1.0"}:
        return "Q1"
    if s in {"Q2", "2", "2.0"}:
        return "Q2"
    if s in {"Q3", "3", "3.0"}:
        return "Q3"
    if s in {"Q4", "4", "4.0"}:
        return "Q4"
    return None


df = pd.DataFrame({
    "song_id": meta[AUDIO_COL].astype(str).str.strip(),
    "lyric_id": meta[LYRIC_COL].astype(str).str.strip(),
    "label": meta[LABEL_COL].map(normalize_label),
})
if ARTIST_COL:
    df["artist"] = meta[ARTIST_COL].fillna("").astype(str).str.strip()
if TITLE_COL:
    df["title"] = meta[TITLE_COL].fillna("").astype(str).str.strip()


def find_media_file(song_id, label, roots, extensions, subdirs):
    raw = Path(str(song_id))
    stem = raw.stem if raw.suffix else raw.name
    candidates = []
    for root in roots:
        for sub in subdirs:
            base = root / sub if sub else root
            candidates.append(base / f"{stem}{raw.suffix}" if raw.suffix else None)
            for ext in extensions:
                candidates.append(base / f"{stem}{ext}")
    for p in candidates:
        if p is not None and p.exists():
            return str(p)
    return None


df["audio_path"] = [
    find_media_file(s, l, AUDIO_ROOTS, [".mp3", ".wav", ".flac", ".ogg", ".m4a"],
                    ["", l, l.lower()])
    for s, l in zip(df.song_id, df.label.fillna("Q1"))
]
df["lyrics_path"] = [
    find_media_file(s, l, LYRIC_ROOTS, [".txt"],
                    ["", l, l.lower()])
    for s, l in zip(df.lyric_id, df.label.fillna("Q1"))
]
df["has_audio"] = df.audio_path.notna()
df["has_lyrics"] = df.lyrics_path.notna()

print(f"Audio ditemukan: {df.has_audio.sum()} / {len(df)}")
print(f"Lirik ditemukan: {df.has_lyrics.sum()} / {len(df)}")
if (~df.has_audio).any():
    print(df.loc[~df.has_audio, ["song_id", "lyric_id", "label"]].head(10).to_string())
if (~df.has_lyrics).any():
    print(df.loc[~df.has_lyrics, ["song_id", "lyric_id", "label"]].head(10).to_string())

## 3. Exploratory Data Analysis

In [ ]:
print("Distribusi quadrant (seluruh metadata):")
print(df.label.value_counts(dropna=False).sort_index().to_string())
print("\nPersentase:")
print((df.label.value_counts(normalize=True).sort_index() * 100).round(2).to_string())

fig, ax = plt.subplots(figsize=(5.5, 3.6))
counts = df.label.value_counts().reindex(CLASS_ORDER).fillna(0)
ax.bar(counts.index, counts.values, color="#4C78A8")
for i, v in enumerate(counts.values):
    ax.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Jumlah lagu")
ax.set_title("Distribusi quadrant MERGE BC")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "eda_class_distribution.png", dpi=160)
plt.show()

# Duplikasi dan kelengkapan pasangan
print("\nDuplikasi song_id :", int(df.song_id.duplicated().sum()))
print("Duplikasi lyric_id:", int(df.lyric_id.duplicated().sum()))
if "artist" in df and "title" in df:
    key = (df.artist.str.lower().str.strip() + " | " + df.title.str.lower().str.strip())
    print("Duplikasi artist+title:", int(key.duplicated().sum()))

if "Duration" in meta.columns:
    dur = pd.to_numeric(meta["Duration"], errors="coerce")
    print("\nDurasi penuh lagu dari metadata (bukan durasi klip):")
    print(dur.describe().round(2).to_string())
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.hist(dur.dropna(), bins=40, color="#54A24B")
    ax.set_xlabel("Durasi lagu (detik)")
    ax.set_ylabel("Frekuensi")
    ax.set_title("Distribusi durasi metadata")
    fig.tight_layout()
    fig.savefig(FIG / "eda_duration.png", dpi=160)
    plt.show()

## 4. Pemeriksaan kualitas dan konsistensi dataset

Pemeriksaan mencakup file audio/lirik, format dan sample rate, channel, durasi klip,
lirik kosong, label tidak valid, duplikasi, dan pasangan yang tidak lengkap.
Setiap lagu yang dibuang dicatat beserta alasannya.

In [ ]:
import soundfile as sf

if not AUDIO_ROOTS:
    raise FileNotFoundError("Folder audio tidak ditemukan.")
if not LYRIC_ROOTS:
    raise FileNotFoundError("Folder lyrics tidak ditemukan.")


def read_text(path):
    try:
        return Path(path).read_text(encoding="utf-8", errors="ignore")
    except Exception as exc:
        return f"__READ_ERROR__:{exc}"


def clean_lyrics(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


audio_rows = []
for i, row in df.iterrows():
    info = {"sr": np.nan, "channels": np.nan, "clip_duration": np.nan,
            "format": None, "audio_error": None}
    if not row.audio_path:
        info["audio_error"] = "file audio tidak ditemukan"
    else:
        try:
            sf_info = sf.info(row.audio_path)
            info.update({
                "sr": sf_info.samplerate,
                "channels": sf_info.channels,
                "clip_duration": sf_info.frames / float(sf_info.samplerate),
                "format": sf_info.format,
            })
            # Uji decode singkat; menangkap file yang info-nya valid tetapi corrupt.
            sf.read(row.audio_path, frames=min(4096, sf_info.frames), dtype="float32")
        except Exception as exc:
            info["audio_error"] = f"tidak dapat dibaca: {exc}"
    audio_rows.append(info)

audio_info = pd.DataFrame(audio_rows, index=df.index)
df = pd.concat([df, audio_info], axis=1)

df["lyrics_text"] = [clean_lyrics(read_text(p)) if p else "" for p in df.lyrics_path]
df["lyrics_chars"] = df.lyrics_text.str.len()
df["lyrics_read_error"] = df.lyrics_text.str.startswith("__READ_ERROR__")

reasons = []
for i, row in df.iterrows():
    r = []
    if not row.has_audio:
        r.append("audio hilang")
    if not row.has_lyrics:
        r.append("lirik hilang")
    if row.audio_error:
        r.append(str(row.audio_error))
    if pd.notna(row.clip_duration) and row.clip_duration < MIN_AUDIO_SEC:
        r.append(f"audio < {MIN_AUDIO_SEC:.2f} detik")
    if row.lyrics_read_error:
        r.append("lirik tidak dapat dibaca")
    elif row.lyrics_chars < 100:
        r.append("lirik < 100 karakter")
    if row.label is None:
        r.append("label tidak valid")
    reasons.append("; ".join(dict.fromkeys(r)))

df["qc_reason"] = reasons
df["usable"] = df.qc_reason.eq("")

# Satu baris per song: duplikasi song_id persis dibuang, bukan dirata-ratakan.
dup_song = df.song_id.duplicated(keep="first")
df.loc[dup_song & df.usable, "qc_reason"] = "song_id duplikat"
df.loc[dup_song, "usable"] = False

print("Ringkasan QC:")
print(f"  total metadata          : {len(df)}")
print(f"  audio hilang            : {int((~df.has_audio).sum())}")
print(f"  lirik hilang            : {int((~df.has_lyrics).sum())}")
print(f"  audio tidak terbaca     : {int(df.audio_error.notna().sum())}")
print(f"  audio terlalu pendek    : {int((df.clip_duration < MIN_AUDIO_SEC).sum())}")
print(f"  lirik < 100 karakter    : {int((df.lyrics_chars < 100).sum())}")
print(f"  label tidak valid       : {int(df.label.isna().sum())}")
print(f"  duplikasi song_id       : {int(dup_song.sum())}")
print(f"  usable                  : {int(df.usable.sum())}")

if df.usable.sum() == 0:
    raise RuntimeError("Tidak ada data usable setelah QC. Periksa path audio/lirik.")

# Group-aware: song_id, lyric_id yang dipakai bersama, dan artist+title yang sama
# digabung agar tidak terpisah antar-subset.
def build_group_ids(frame):
    frame = frame.reset_index(drop=True)
    parent = list(range(len(frame)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    keys = []
    keys.append(("lyric_id", frame.lyric_id.astype(str)))
    if "artist" in frame and "title" in frame:
        key = (frame.artist.fillna("").str.lower().str.strip() + "|" +
               frame.title.fillna("").str.lower().str.strip())
        keys.append(("artist_title", key))
    for _, values in keys:
        for _, idxs in frame.groupby(values).groups.items():
            idxs = list(idxs)
            if len(idxs) > 1:
                for j in idxs[1:]:
                    union(idxs[0], j)
    root_to_gid = {}
    out = []
    for i in range(len(frame)):
        r = find(i)
        if r not in root_to_gid:
            root_to_gid[r] = f"G{len(root_to_gid):05d}"
        out.append(root_to_gid[r])
    return out


df_clean = df[df.usable].copy().reset_index(drop=True)
df_clean["group_id"] = build_group_ids(df_clean)
df_clean["label_idx"] = df_clean.label.map(LABEL_TO_IDX).astype(int)

if QUICK_RUN:
    # Subset stratified kecil; hanya untuk smoke test, bukan hasil penelitian.
    parts = []
    per_class = max(2, QUICK_N // len(CLASS_ORDER))
    for _, grp in df_clean.groupby("label"):
        parts.append(grp.sample(n=min(len(grp), per_class), random_state=SEED))
    df_clean = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df_clean["group_id"] = build_group_ids(df_clean)
    df_clean["label_idx"] = df_clean.label.map(LABEL_TO_IDX).astype(int)

N = len(df_clean)
Y = df_clean.label_idx.to_numpy()
if N == 0:
    raise RuntimeError("Tidak ada data setelah filtering.")

print(f"\nJumlah sampel final: {N}")
print(df_clean.label.value_counts().reindex(CLASS_ORDER).to_string())

# Simpan ringkasan dataset dan exclusion log
summary_rows = [
    ("total_metadata", len(df)),
    ("audio_missing", int((~df.has_audio).sum())),
    ("lyrics_missing", int((~df.has_lyrics).sum())),
    ("audio_unreadable", int(df.audio_error.notna().sum())),
    ("audio_too_short", int((df.clip_duration < MIN_AUDIO_SEC).sum())),
    ("lyrics_too_short", int((df.lyrics_chars < 100).sum())),
    ("invalid_label", int(df.label.isna().sum())),
    ("duplicate_song_id", int(dup_song.sum())),
    ("final_samples", N),
]
for c in CLASS_ORDER:
    summary_rows.append((f"final_{c}", int((df_clean.label == c).sum())))
pd.DataFrame(summary_rows, columns=["metric", "value"]).to_csv(EXP / "dataset_summary.csv", index=False)
df[~df.usable][["song_id", "lyric_id", "label", "qc_reason", "audio_path", "lyrics_path"]].to_csv(
    EXP / "dataset_exclusions.csv", index=False
)
df_clean.to_csv(PROC / "manifest_qc.csv", index=False)
print("\nDataset summary ->", EXP / "dataset_summary.csv")
print("Exclusion log   ->", EXP / "dataset_exclusions.csv")

## 5. Preprocessing audio dan lirik

Audio diubah ke mono dan di-resample ke 24 kHz untuk MERT. Tidak ada normalisasi
tambahan yang menghilangkan karakteristik musik; preprocessing mengikuti kebutuhan checkpoint.
Lirik dibersihkan secara minimal, mempertahankan huruf besar/kecil dan tanda baca.

In [ ]:
import librosa


def load_audio_mono(path, target_sr):
    y, sr = sf.read(path, dtype="float32", always_2d=True)
    y = y.mean(axis=1)
    if sr != target_sr:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=target_sr)
    return np.asarray(y, dtype=np.float32)


def pad_or_crop(y, sr, max_sec):
    target = int(round(float(max_sec) * sr))
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return np.asarray(y, dtype=np.float32)


print("Sample rate audio:")
print(df_clean.sr.value_counts(dropna=False).sort_index().to_string())
print("\nChannel audio:")
print(df_clean.channels.value_counts(dropna=False).sort_index().to_string())
print("\nFormat audio:")
print(df_clean.format.value_counts(dropna=False).to_string())
print("\nDurasi klip (detik):")
print(df_clean.clip_duration.describe().round(2).to_string())
print("\nPanjang lirik (karakter):")
print(df_clean.lyrics_chars.describe().round(1).to_string())

## 6. Ekstraksi fitur MERT

Representasi yang disimpan:
- `mert_pooled.npy` — mean pooling layer terakhir, 768 dimensi.
- `mert_seq.npy` — sequence layer terakhir, di-downsample ke 128 langkah waktu.
- `mert_last4_mean.npy` — opsional, rata-rata empat layer terakhir.
- `mert_last4_concat.npy` — opsional, concatenation empat layer terakhir.

Catatan: `AutoModel` MERT-v1-95M pada lingkungan ini tidak mengembalikan `hidden_states`
lewat `output_hidden_states`; opsi multi-layer dinonaktifkan secara eksplisit agar tidak
menyimpan cache nol.

Cache memakai memmap dan `done` mask sehingga ekstraksi dapat dilanjutkan.

In [ ]:
MERT_META = FEAT / "mert_meta.json"
MERT_POOLED_PATH = FEAT / "mert_pooled.npy"
MERT_SEQ_PATH = FEAT / "mert_seq.npy"
MERT_LAST4_MEAN_PATH = FEAT / "mert_last4_mean.npy"
MERT_LAST4_CONCAT_PATH = FEAT / "mert_last4_concat.npy"
MERT_DONE_PATH = FEAT / "mert_done.npy"
SONGS_HASH = hashlib.md5("\n".join(df_clean.song_id.astype(str)).encode("utf-8")).hexdigest()

mert_meta_expected = {
    "model_id": MERT_ID,
    "sample_rate": MERT_SR,
    "max_audio_sec": MAX_AUDIO_SEC,
    "seq_len": AUDIO_SEQ_LEN,
    "hidden_size": AUDIO_DIM,
    "songs_hash": SONGS_HASH,
    "n": N,
}

# Cache dianggap valid jika bentuk tensor cocok. Metadata dicek bila ada; cache parsial
# dari ekstraksi yang terputus tetap dapat dilanjutkan.
mert_shape_ok = False
if all(p.exists() for p in [MERT_POOLED_PATH, MERT_SEQ_PATH, MERT_DONE_PATH]):
    try:
        _a = np.load(MERT_POOLED_PATH, mmap_mode="r")
        _b = np.load(MERT_SEQ_PATH, mmap_mode="r")
        _d = np.load(MERT_DONE_PATH)
        mert_shape_ok = (
            _a.shape == (N, AUDIO_DIM)
            and _b.shape == (N, AUDIO_SEQ_LEN, AUDIO_DIM)
            and _d.shape == (N,)
        )
    except Exception:
        mert_shape_ok = False

mert_cache_ok = False
if not FORCE_REEXTRACT:
    if MERT_META.exists():
        try:
            old = json.loads(MERT_META.read_text())
            mert_cache_ok = mert_shape_ok and all(
                old.get(k) == v for k, v in mert_meta_expected.items()
            )
        except Exception:
            mert_cache_ok = False
    elif mert_shape_ok:
        mert_cache_ok = True

if not mert_cache_ok:
    for p in [MERT_POOLED_PATH, MERT_SEQ_PATH, MERT_LAST4_MEAN_PATH,
              MERT_LAST4_CONCAT_PATH, MERT_DONE_PATH, MERT_META]:
        if p.exists():
            p.unlink()

mert_pooled = np.lib.format.open_memmap(
    MERT_POOLED_PATH, mode="r+" if MERT_POOLED_PATH.exists() else "w+",
    dtype=np.float16, shape=(N, AUDIO_DIM)
)
mert_seq = np.lib.format.open_memmap(
    MERT_SEQ_PATH, mode="r+" if MERT_SEQ_PATH.exists() else "w+",
    dtype=np.float16, shape=(N, AUDIO_SEQ_LEN, AUDIO_DIM)
)
mert_done = np.load(MERT_DONE_PATH) if MERT_DONE_PATH.exists() else np.zeros(N, dtype=bool)

if RUN_MERT_MULTILAYER:
    # AutoModel MERT-v1-95M tidak mengekspos hidden_states lewat output_hidden_states;
    # eksperimen multi-layer memerlukan forward hook. Flag dinonaktifkan agar tidak
    # menghasilkan cache nol yang menyesatkan.
    print("PERINGATAN: multi-layer MERT belum didukung checkpoint ini; flag dinonaktifkan.")
    RUN_MERT_MULTILAYER = False

if RUN_MERT_MULTILAYER:
    mert_last4_mean = np.lib.format.open_memmap(
        MERT_LAST4_MEAN_PATH, mode="r+" if MERT_LAST4_MEAN_PATH.exists() else "w+",
        dtype=np.float16, shape=(N, AUDIO_DIM)
    )
    mert_last4_concat = np.lib.format.open_memmap(
        MERT_LAST4_CONCAT_PATH, mode="r+" if MERT_LAST4_CONCAT_PATH.exists() else "w+",
        dtype=np.float16, shape=(N, 4 * AUDIO_DIM)
    )
else:
    mert_last4_mean = None
    mert_last4_concat = None

print(f"Cache MERT: {int(mert_done.sum())}/{N} selesai.")
if not mert_done.all():
    print("Ekstraksi MERT dimulai/dilanjutkan...")
    from transformers import AutoModel, Wav2Vec2FeatureExtractor
    processor = Wav2Vec2FeatureExtractor.from_pretrained(MERT_ID, trust_remote_code=True)
    mert_model = AutoModel.from_pretrained(MERT_ID, trust_remote_code=True).to(DEVICE).eval()
    hidden_size = int(getattr(mert_model.config, "hidden_size", AUDIO_DIM))
    if hidden_size != AUDIO_DIM:
        raise ValueError(f"Hidden size MERT {hidden_size} != AUDIO_DIM {AUDIO_DIM}")

    t0 = time.time()
    for i, row in enumerate(df_clean.itertuples()):
        if mert_done[i]:
            continue
        try:
            y = load_audio_mono(row.audio_path, MERT_SR)
            y = pad_or_crop(y, MERT_SR, MAX_AUDIO_SEC)
            inputs = processor(y, sampling_rate=MERT_SR, return_tensors="pt")
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.inference_mode():
                out = mert_model(**inputs, output_hidden_states=True)
            last = out.last_hidden_state[0].detach().float().cpu()
            mert_pooled[i] = last.mean(dim=0).numpy().astype(np.float16)
            seq = F.adaptive_avg_pool1d(
                last.transpose(0, 1).unsqueeze(0), AUDIO_SEQ_LEN
            ).squeeze(0).transpose(0, 1)
            mert_seq[i] = seq.numpy().astype(np.float16)
            if mert_last4_mean is not None and out.hidden_states is not None:
                hs = torch.stack([h[0].detach().float().cpu() for h in out.hidden_states[-4:]], 0)
                per_layer = hs.mean(dim=1)
                mert_last4_mean[i] = per_layer.mean(0).numpy().astype(np.float16)
                mert_last4_concat[i] = per_layer.reshape(-1).numpy().astype(np.float16)
            mert_done[i] = True
        except Exception as exc:
            raise RuntimeError(f"Gagal ekstraksi MERT untuk {row.song_id}: {exc}") from exc
        if (i + 1) % 10 == 0 or i == N - 1:
            np.save(MERT_DONE_PATH, mert_done)
            print(f"  MERT {i + 1}/{N} | {time.time() - t0:.1f}s", flush=True)
    np.save(MERT_DONE_PATH, mert_done)
    MERT_META.write_text(json.dumps({
        **mert_meta_expected,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "complete": bool(mert_done.all()),
    }, indent=2))
    del mert_model, processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Ekstraksi MERT selesai dalam {time.time() - t0:.1f}s")
else:
    if not MERT_META.exists():
        MERT_META.write_text(json.dumps({
            **mert_meta_expected,
            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "complete": True,
        }, indent=2))

mert_pooled.flush()
mert_seq.flush()
mert_seq.flush()

## 7. Ekstraksi fitur RoBERTa

Seluruh lirik dipakai, ditruncate pada 512 token. Mean pooling memperhatikan attention mask.
Disimpan pooled embedding 1024 dimensi dan sequence 128 langkah untuk model sekuensial.

In [ ]:
ROBERTA_META = FEAT / "roberta_meta.json"
ROBERTA_POOLED_PATH = FEAT / "roberta_pooled.npy"
ROBERTA_SEQ_PATH = FEAT / "roberta_seq.npy"
ROBERTA_IDS_PATH = FEAT / "roberta_ids.npy"
ROBERTA_MASK_PATH = FEAT / "roberta_mask.npy"
ROBERTA_DONE_PATH = FEAT / "roberta_done.npy"

roberta_meta_expected = {
    "model_id": ROBERTA_ID,
    "max_tokens": MAX_LYRICS_TOKENS,
    "seq_len": TEXT_SEQ_LEN,
    "songs_hash": SONGS_HASH,
    "n": N,
    "hidden_size": TEXT_DIM,
}

# Bila cache lama ada, pakai model dan dimensi yang tercatat agar konsisten.
if ROBERTA_META.exists():
    try:
        old = json.loads(ROBERTA_META.read_text())
        ROBERTA_ID = old.get("model_id", ROBERTA_ID)
        TEXT_DIM = int(old.get("hidden_size", TEXT_DIM))
        roberta_meta_expected["model_id"] = ROBERTA_ID
        roberta_meta_expected["hidden_size"] = TEXT_DIM
    except Exception:
        pass

roberta_shape_ok = False
if all(p.exists() for p in [ROBERTA_POOLED_PATH, ROBERTA_SEQ_PATH, ROBERTA_IDS_PATH,
                            ROBERTA_MASK_PATH, ROBERTA_DONE_PATH]):
    try:
        _p = np.load(ROBERTA_POOLED_PATH, mmap_mode="r")
        _q = np.load(ROBERTA_SEQ_PATH, mmap_mode="r")
        _i = np.load(ROBERTA_IDS_PATH, mmap_mode="r")
        _m = np.load(ROBERTA_MASK_PATH, mmap_mode="r")
        _d = np.load(ROBERTA_DONE_PATH)
        roberta_shape_ok = (
            _p.shape == (N, TEXT_DIM)
            and _q.shape == (N, TEXT_SEQ_LEN, TEXT_DIM)
            and _i.shape == (N, MAX_LYRICS_TOKENS)
            and _m.shape == (N, MAX_LYRICS_TOKENS)
            and _d.shape == (N,)
        )
    except Exception:
        roberta_shape_ok = False

roberta_cache_ok = False
if not FORCE_REEXTRACT:
    if ROBERTA_META.exists():
        try:
            old = json.loads(ROBERTA_META.read_text())
            roberta_cache_ok = roberta_shape_ok and all(
                old.get(k) == v for k, v in roberta_meta_expected.items()
            )
        except Exception:
            roberta_cache_ok = False
    elif roberta_shape_ok:
        roberta_cache_ok = True

if not roberta_cache_ok:
    for p in [ROBERTA_POOLED_PATH, ROBERTA_SEQ_PATH, ROBERTA_IDS_PATH,
              ROBERTA_MASK_PATH, ROBERTA_DONE_PATH, ROBERTA_META]:
        if p.exists():
            p.unlink()

roberta_pooled = np.lib.format.open_memmap(
    ROBERTA_POOLED_PATH, mode="r+" if ROBERTA_POOLED_PATH.exists() else "w+",
    dtype=np.float16, shape=(N, TEXT_DIM)
)
roberta_seq = np.lib.format.open_memmap(
    ROBERTA_SEQ_PATH, mode="r+" if ROBERTA_SEQ_PATH.exists() else "w+",
    dtype=np.float16, shape=(N, TEXT_SEQ_LEN, TEXT_DIM)
)
roberta_ids = np.lib.format.open_memmap(
    ROBERTA_IDS_PATH, mode="r+" if ROBERTA_IDS_PATH.exists() else "w+",
    dtype=np.int32, shape=(N, MAX_LYRICS_TOKENS)
)
roberta_mask = np.lib.format.open_memmap(
    ROBERTA_MASK_PATH, mode="r+" if ROBERTA_MASK_PATH.exists() else "w+",
    dtype=np.bool_, shape=(N, MAX_LYRICS_TOKENS)
)
roberta_done = np.load(ROBERTA_DONE_PATH) if ROBERTA_DONE_PATH.exists() else np.zeros(N, dtype=bool)
pending = [i for i in range(N) if not roberta_done[i]]
print(f"Cache RoBERTa: {int(roberta_done.sum())}/{N} selesai.")
if pending:
    print("Ekstraksi RoBERTa dimulai/dilanjutkan...")
    from transformers import AutoTokenizer, AutoModel
    try:
        tokenizer = AutoTokenizer.from_pretrained(ROBERTA_ID, use_fast=True)
        roberta_model = AutoModel.from_pretrained(ROBERTA_ID).to(DEVICE).eval()
    except Exception as exc:
        print(f"Gagal memuat {ROBERTA_ID}: {exc}")
        print(f"Mencoba fallback {ROBERTA_FALLBACK_ID}...")
        ROBERTA_ID = ROBERTA_FALLBACK_ID
        tokenizer = AutoTokenizer.from_pretrained(ROBERTA_FALLBACK_ID, use_fast=True)
        roberta_model = AutoModel.from_pretrained(ROBERTA_FALLBACK_ID).to(DEVICE).eval()
    actual_dim = int(roberta_model.config.hidden_size)
    if actual_dim != TEXT_DIM:
        # Dimensi berubah (mis. fallback base). Cache lama tidak dapat dipakai.
        TEXT_DIM = actual_dim
        for p in [ROBERTA_POOLED_PATH, ROBERTA_SEQ_PATH, ROBERTA_IDS_PATH,
                  ROBERTA_MASK_PATH, ROBERTA_DONE_PATH, ROBERTA_META]:
            if p.exists():
                p.unlink()
        roberta_pooled = np.lib.format.open_memmap(ROBERTA_POOLED_PATH, mode="w+", dtype=np.float16, shape=(N, TEXT_DIM))
        roberta_seq = np.lib.format.open_memmap(ROBERTA_SEQ_PATH, mode="w+", dtype=np.float16, shape=(N, TEXT_SEQ_LEN, TEXT_DIM))
        roberta_ids = np.lib.format.open_memmap(ROBERTA_IDS_PATH, mode="w+", dtype=np.int32, shape=(N, MAX_LYRICS_TOKENS))
        roberta_mask = np.lib.format.open_memmap(ROBERTA_MASK_PATH, mode="w+", dtype=np.bool_, shape=(N, MAX_LYRICS_TOKENS))
        roberta_done = np.zeros(N, dtype=bool)
        pending = list(range(N))
    roberta_meta_expected["model_id"] = ROBERTA_ID
    roberta_meta_expected["hidden_size"] = TEXT_DIM
    batch_size = ROBERTA_BATCH
    t0 = time.time()
    for start in range(0, len(pending), batch_size):
        idxs = pending[start:start + batch_size]
        texts = [df_clean.loc[i, "lyrics_text"] for i in idxs]
        enc = tokenizer(
            texts, padding="max_length", truncation=True,
            max_length=MAX_LYRICS_TOKENS, return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode():
            out = roberta_model(**enc)
        h = out.last_hidden_state.detach().float().cpu()
        mask = enc["attention_mask"].detach().cpu().bool()
        pooled = (h * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        for j, idx in enumerate(idxs):
            L = int(mask[j].sum())
            hj = h[j, :L, :]
            sj = F.adaptive_avg_pool1d(
                hj.transpose(0, 1).unsqueeze(0), TEXT_SEQ_LEN
            ).squeeze(0).transpose(0, 1)
            roberta_pooled[idx] = pooled[j].numpy().astype(np.float16)
            roberta_seq[idx] = sj.numpy().astype(np.float16)
            roberta_ids[idx] = enc["input_ids"][j].detach().cpu().numpy()
            roberta_mask[idx] = mask[j].numpy()
            roberta_done[idx] = True
        if (start + len(idxs)) % 50 == 0 or start + len(idxs) == N:
            print(f"  RoBERTa {start + len(idxs)}/{N} | {time.time() - t0:.1f}s", flush=True)
    np.save(ROBERTA_DONE_PATH, roberta_done)
    ROBERTA_META.write_text(json.dumps({
        **roberta_meta_expected,
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "complete": bool(roberta_done.all()),
    }, indent=2))
    del roberta_model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Ekstraksi RoBERTa selesai dalam {time.time() - t0:.1f}s")
else:
    if not ROBERTA_META.exists():
        ROBERTA_META.write_text(json.dumps({
            **roberta_meta_expected,
            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "complete": True,
        }, indent=2))

print("TEXT_DIM =", TEXT_DIM)

## 8. Validasi cache dan Dataset PyTorch

In [ ]:
A_POOL = np.load(MERT_POOLED_PATH, mmap_mode="r")
A_SEQ = np.load(MERT_SEQ_PATH, mmap_mode="r")
L_POOL = np.load(ROBERTA_POOLED_PATH, mmap_mode="r")
L_SEQ = np.load(ROBERTA_SEQ_PATH, mmap_mode="r")
L_IDS = np.load(ROBERTA_IDS_PATH, mmap_mode="r")
L_MASK = np.load(ROBERTA_MASK_PATH, mmap_mode="r")

assert A_POOL.shape == (N, AUDIO_DIM), (A_POOL.shape, N, AUDIO_DIM)
assert A_SEQ.shape == (N, AUDIO_SEQ_LEN, AUDIO_DIM), A_SEQ.shape
assert L_POOL.shape == (N, TEXT_DIM), (L_POOL.shape, N, TEXT_DIM)
assert L_SEQ.shape == (N, TEXT_SEQ_LEN, TEXT_DIM), L_SEQ.shape
print("A_POOL", A_POOL.shape, A_POOL.dtype)
print("A_SEQ ", A_SEQ.shape, A_SEQ.dtype)
print("L_POOL", L_POOL.shape, L_POOL.dtype)
print("L_SEQ ", L_SEQ.shape, L_SEQ.dtype)


class FeatureDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        return {
            "audio_pool": torch.from_numpy(np.asarray(A_POOL[idx], dtype=np.float32)),
            "lyrics_pool": torch.from_numpy(np.asarray(L_POOL[idx], dtype=np.float32)),
            "audio_seq": torch.from_numpy(np.asarray(A_SEQ[idx], dtype=np.float32)),
            "lyrics_seq": torch.from_numpy(np.asarray(L_SEQ[idx], dtype=np.float32)),
            "label": torch.tensor(int(Y[idx]), dtype=torch.long),
        }


def make_loader(indices, batch_size, shuffle=False):
    return DataLoader(
        FeatureDataset(indices), batch_size=int(batch_size), shuffle=shuffle,
        num_workers=0, pin_memory=torch.cuda.is_available(), drop_last=False
    )

## 9. Pembuatan split 70/15/15 dan 40/30/30

Split resmi MERGE dicoba sebagai repeat pertama. Bila split resmi memecah group duplikat
(song/lyric/artist-title), notebook otomatis memakai split generated yang stratified dan
group-aware; keputusan ini tercatat di `split_summary.csv`. Repeat berikutnya dibuat dengan
seed berbeda. Semua model pada repeat yang sama memakai split yang sama; test set tidak
pernah dipakai untuk tuning.

In [ ]:
def find_split_files(scenario):
    for root in SPLIT_ROOTS:
        for d in root.rglob("*"):
            if d.is_dir() and scenario in d.name:
                csvs = sorted(d.glob("*.csv"))
                if csvs:
                    return d, csvs
    return None, []


def load_official_split(scenario):
    folder, csvs = find_split_files(scenario)
    if not csvs:
        return None
    frames = []
    for p in csvs:
        name = p.name.lower()
        if "train" in name:
            part = "train"
        elif "valid" in name or "val" in name:
            part = "validation"
        elif "test" in name:
            part = "test"
        else:
            continue
        s = pd.read_csv(p, dtype=str)
        s.columns = [c.strip() for c in s.columns]
        song_col = next((c for c in s.columns if c.lower() in ("song", "song_id", "audio_song")), None)
        lab_col = next((c for c in s.columns if c.lower() in ("quadrant", "label")), None)
        if song_col is None:
            continue
        s = s.rename(columns={song_col: "song_id"})
        if lab_col:
            s = s.rename(columns={lab_col: "label"})
        s["split"] = part
        frames.append(s[["song_id", "split"] + (["label"] if "label" in s.columns else [])])
    if not frames:
        return None
    split_df = pd.concat(frames, ignore_index=True)
    split_df["song_id"] = split_df.song_id.astype(str).str.strip()
    mapping = dict(zip(split_df.song_id, split_df.split))
    missing = df_clean.loc[~df_clean.song_id.isin(mapping), "song_id"].tolist()
    if len(missing) > 0.05 * N:
        print(f"Split resmi {scenario} tidak mencakup {len(missing)} lagu; fallback ke split generated.")
        return None
    idx = {"train": [], "validation": [], "test": []}
    for i, sid in enumerate(df_clean.song_id):
        part = mapping.get(sid)
        if part in idx:
            idx[part].append(i)
    out = {k: np.asarray(v, dtype=np.int64) for k, v in idx.items()}
    # Split resmi tidak boleh memecah group (song/lyric/artist-title) antar-subset.
    _tmp = pd.concat([
        df_clean.loc[out["train"], ["group_id"]].assign(split="train"),
        df_clean.loc[out["validation"], ["group_id"]].assign(split="validation"),
        df_clean.loc[out["test"], ["group_id"]].assign(split="test"),
    ])
    if _tmp.groupby("group_id").split.nunique().max() > 1:
        print(f"Split resmi {scenario} memecah group; memakai split generated.")
        return None
    for k, v in out.items():
        missing = set(CLASS_ORDER) - set(df_clean.loc[v, "label"])
        if missing:
            print(f"Split resmi {scenario}/{k} tidak punya kelas {missing}; memakai split generated.")
            return None
    return out


def make_group_split(scenario, seed, max_attempts=30):
    frac_train, frac_val, frac_test = SCENARIOS[scenario]
    groups = df_clean.groupby("group_id", as_index=False).agg(label=("label", "first"))
    last = None
    for attempt in range(max_attempts):
        rs = seed + attempt
        train_g, temp_g = train_test_split(
            groups, test_size=frac_val + frac_test, stratify=groups.label, random_state=rs
        )
        val_share = frac_val / (frac_val + frac_test)
        val_g, test_g = train_test_split(
            temp_g, test_size=1.0 - val_share, stratify=temp_g.label, random_state=rs
        )
        out = {}
        for name, g in [("train", train_g), ("validation", val_g), ("test", test_g)]:
            ids = set(g.group_id)
            out[name] = df_clean.index[df_clean.group_id.isin(ids)].to_numpy(dtype=np.int64)
        last = out
        if all(set(CLASS_ORDER).issubset(set(df_clean.loc[v, "label"])) for v in out.values()):
            return out
    print(f"PERINGATAN: split {scenario} belum mencakup semua kelas setelah {max_attempts} percobaan.")
    return last


def validate_split(split, name=""):
    sets = {k: set(v.tolist()) for k, v in split.items()}
    assert sets["train"].isdisjoint(sets["validation"]), f"{name}: train/val overlap"
    assert sets["train"].isdisjoint(sets["test"]), f"{name}: train/test overlap"
    assert sets["validation"].isdisjoint(sets["test"]), f"{name}: val/test overlap"
    _tmp = pd.concat([
        df_clean.loc[split["train"], ["group_id"]].assign(split="train"),
        df_clean.loc[split["validation"], ["group_id"]].assign(split="validation"),
        df_clean.loc[split["test"], ["group_id"]].assign(split="test"),
    ])
    if _tmp.groupby("group_id").split.nunique().max() > 1:
        raise ValueError(f"{name}: group bocor antar-subset")
    for k, v in split.items():
        if len(v) == 0:
            raise ValueError(f"{name}: subset {k} kosong")
        missing = set(CLASS_ORDER) - set(df_clean.loc[v, "label"])
        if missing:
            msg = f"{name}/{k}: kelas tidak ada {missing}"
            if STRICT_CLASS_COVERAGE:
                raise ValueError(msg + ". Perbesar QUICK_N atau gunakan split full.")
            print("PERINGATAN " + msg)


SPLITS = {}
split_rows = []
for scenario in SCENARIOS:
    official = load_official_split(scenario)
    for rep in range(1, REPEATS + 1):
        if rep == 1 and official is not None:
            split = official
            source = "official"
            split_seed = SEED
        else:
            split_seed = SEED + rep - 1
            split = make_group_split(scenario, split_seed)
            source = "generated"
        validate_split(split, f"{scenario}/rep{rep}")
        SPLITS[(scenario, rep)] = {
            "train": split["train"], "validation": split["validation"], "test": split["test"],
            "seed": split_seed, "source": source,
        }
        for part in ["train", "validation", "test"]:
            counts = df_clean.loc[split[part], "label"].value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
            split_rows.append({
                "scenario": scenario, "repeat": rep, "seed": split_seed, "source": source,
                "split": part, "n": len(split[part]),
                **{c: int(counts[c]) for c in CLASS_ORDER},
            })
        print(f"{scenario} rep{rep} ({source}, seed={split_seed}): "
              f"train={len(split['train'])} val={len(split['validation'])} test={len(split['test'])}")

split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(EXP / "split_summary.csv", index=False)
split_long = []
for (scenario, rep), s in SPLITS.items():
    for part in ["train", "validation", "test"]:
        for i in s[part]:
            split_long.append({
                "scenario": scenario, "repeat": rep, "split": part,
                "song_id": df_clean.loc[i, "song_id"], "group_id": df_clean.loc[i, "group_id"],
                "label": df_clean.loc[i, "label"],
            })
pd.DataFrame(split_long).to_csv(EXP / "splits_long.csv", index=False)
print("\nSplit summary ->", EXP / "split_summary.csv")
print(split_summary.to_string(index=False))

## 10–16. Implementasi model

Setiap arsitektur mendukung tiga modalitas: `audio`, `text`, dan `bimodal`.
Tidak ada satu konfigurasi internal yang dipaksakan ke semua model; classifier,
fusion, dan dimensi tersembunyi mengikuti arsitektur masing-masing.

Catatan adaptasi:
- MoodNet-3 di sini memakai CNN 1D pada sequence MERT/RoBERTa, bukan CNN 2D pada
  mel-spectrogram + GloVe seperti implementasi asli. Ini penyesuaian karena input
  sudah berupa embedding pretrained.
- CRNN BiGRU memakai CNN 1D + BiGRU pada sequence embedding, lalu late fusion.
- Joint Transformer memakai token MERT/RoBERTa, bukan token hasil feature engineering.
- MERT + RoBERTa memakai classifier 768+1024; head 2048 -> 100 milik Wav2Vec2 tidak
  dipaksakan karena Wav2Vec2 dihapus dari lingkup eksperimen.

In [ ]:
MODEL_EARLY = "MERT + RoBERTa (Early Fusion)"
MODEL_MOODNET = "MoodNet-3 (CNN Concatenation)"
MODEL_CRNN = "CRNN BiGRU (Late Fusion)"
MODEL_JOINT = "Joint Transformer (Early Fusion)"
MODEL_WEIGHTED = "Weighted Average Fusion"
MODEL_GATED = "Gated Multimodal Fusion"
MODEL_NAMES = [MODEL_EARLY, MODEL_MOODNET, MODEL_CRNN, MODEL_JOINT, MODEL_WEIGHTED, MODEL_GATED]
MODEL_VARIANTS = [(m, mod) for m in MODEL_NAMES for mod in MODALITIES]


def variant_label(model_name, modality):
    return f"{model_name} | {modality}"


def variant_slug(model_name, modality):
    return f"{slugify(model_name)}__{modality}"


class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, n_classes=4, dropout=0.3):
        super().__init__()
        layers = []
        d = int(in_dim)
        for h in hidden_dims:
            layers += [nn.Linear(d, int(h)), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            d = int(h)
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class MERTRoBERTaEarlyFusion(nn.Module):
    def __init__(self, audio_dim, text_dim, modality="bimodal", hidden=1024,
                 hidden2=256, dropout=0.3, n_classes=4):
        super().__init__()
        self.modality = modality
        if modality == "audio":
            in_dim = audio_dim
        elif modality == "text":
            in_dim = text_dim
        else:
            in_dim = audio_dim + text_dim
        self.net = MLP(in_dim, [hidden, hidden2], n_classes, dropout)

    def forward(self, batch):
        if self.modality == "audio":
            x = batch["audio_pool"]
        elif self.modality == "text":
            x = batch["lyrics_pool"]
        else:
            x = torch.cat([batch["audio_pool"], batch["lyrics_pool"]], dim=1)
        return {"logits": self.net(x)}


class ConvSeqEncoder(nn.Module):
    def __init__(self, in_dim, width=256, dropout=0.3):
        super().__init__()
        channels = [width, width * 2, width * 4]
        layers = []
        c = int(in_dim)
        for ch in channels:
            layers += [
                nn.Conv1d(c, ch, kernel_size=3, padding=1),
                nn.BatchNorm1d(ch),
                nn.ReLU(inplace=True),
                nn.MaxPool1d(2),
                nn.Dropout(dropout),
            ]
            c = ch
        self.net = nn.Sequential(*layers)
        self.out_dim = c

    def forward(self, x):
        # x: (B, T, D) -> (B, D_out)
        x = self.net(x.transpose(1, 2))
        return x.max(dim=-1).values


class MoodNet3(nn.Module):
    def __init__(self, audio_dim, text_dim, modality="bimodal", width=256,
                 hidden=1024, dropout=0.3, n_classes=4):
        super().__init__()
        self.modality = modality
        self.audio = ConvSeqEncoder(audio_dim, width, dropout) if modality in ("audio", "bimodal") else None
        self.text = ConvSeqEncoder(text_dim, width, dropout) if modality in ("text", "bimodal") else None
        if modality == "audio":
            in_dim = self.audio.out_dim
        elif modality == "text":
            in_dim = self.text.out_dim
        else:
            in_dim = self.audio.out_dim + self.text.out_dim
        self.head = MLP(in_dim, [hidden, hidden // 2, hidden // 4], n_classes, dropout)

    def forward(self, batch):
        z = []
        if self.audio is not None:
            z.append(self.audio(batch["audio_seq"]))
        if self.text is not None:
            z.append(self.text(batch["lyrics_seq"]))
        return {"logits": self.head(torch.cat(z, dim=1))}


class CRNNBranch(nn.Module):
    def __init__(self, in_dim, cnn_dim=256, gru_dim=128, dropout=0.3, n_classes=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_dim, cnn_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(cnn_dim, gru_dim, batch_first=True, bidirectional=True)
        self.head = MLP(2 * gru_dim, [128], n_classes, dropout)

    def forward(self, x):
        x = self.cnn(x.transpose(1, 2)).transpose(1, 2)
        h, _ = self.gru(x)
        return self.head(h.mean(dim=1))


class CRNNBiGRULateFusion(nn.Module):
    def __init__(self, audio_dim, text_dim, modality="bimodal", cnn_dim=256,
                 gru_dim=128, dropout=0.3, n_classes=4):
        super().__init__()
        self.modality = modality
        self.audio = CRNNBranch(audio_dim, cnn_dim, gru_dim, dropout, n_classes) if modality in ("audio", "bimodal") else None
        self.text = CRNNBranch(text_dim, cnn_dim, gru_dim, dropout, n_classes) if modality in ("text", "bimodal") else None
        self.fusion_logit = nn.Parameter(torch.zeros(1)) if modality == "bimodal" else None

    def forward(self, batch):
        if self.modality == "audio":
            return {"logits": self.audio(batch["audio_seq"])}
        if self.modality == "text":
            return {"logits": self.text(batch["lyrics_seq"])}
        la = self.audio(batch["audio_seq"])
        ll = self.text(batch["lyrics_seq"])
        w = torch.sigmoid(self.fusion_logit)
        return {
            "logits": w * la + (1.0 - w) * ll,
            "aux_logits": [la, ll],
            "fusion_weight": w.detach(),
        }


class JointTransformer(nn.Module):
    def __init__(self, audio_dim, text_dim, modality="bimodal", d_model=256,
                 n_heads=4, n_layers=2, dropout=0.3, n_classes=4,
                 seq_a=AUDIO_SEQ_LEN, seq_l=TEXT_SEQ_LEN):
        super().__init__()
        self.modality = modality
        self.proj_a = nn.Linear(audio_dim, d_model) if modality in ("audio", "bimodal") else None
        self.proj_l = nn.Linear(text_dim, d_model) if modality in ("text", "bimodal") else None
        self.mod = nn.Parameter(torch.zeros(2, d_model))
        nn.init.trunc_normal_(self.mod, std=0.02)
        max_tokens = (seq_a if modality == "audio" else seq_l if modality == "text" else seq_a + seq_l)
        self.pos = nn.Parameter(torch.zeros(1, max_tokens, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, 4 * d_model, dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.enc = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = MLP(d_model, [256, 128], n_classes, dropout)

    def forward(self, batch):
        if self.modality == "audio":
            x = self.proj_a(batch["audio_seq"]) + self.mod[0]
        elif self.modality == "text":
            x = self.proj_l(batch["lyrics_seq"]) + self.mod[1]
        else:
            a = self.proj_a(batch["audio_seq"]) + self.mod[0]
            l = self.proj_l(batch["lyrics_seq"]) + self.mod[1]
            x = torch.cat([a, l], dim=1)
        x = x + self.pos[:, : x.size(1)]
        x = self.norm(self.enc(x))
        return {"logits": self.head(x.mean(dim=1))}


class WeightedAverageFusion(nn.Module):
    def __init__(self, audio_dim, text_dim, modality="bimodal", hidden=256,
                 dropout=0.3, n_classes=4):
        super().__init__()
        self.modality = modality
        self.audio_head = MLP(audio_dim, [hidden, hidden // 2], n_classes, dropout) if modality in ("audio", "bimodal") else None
        self.lyrics_head = MLP(text_dim, [hidden, hidden // 2], n_classes, dropout) if modality in ("text", "bimodal") else None
        self.register_buffer("alpha", torch.tensor(0.5))

    def set_alpha(self, value):
        self.alpha.fill_(float(value))

    def forward(self, batch):
        if self.modality == "audio":
            return {"logits": self.audio_head(batch["audio_pool"])}
        if self.modality == "text":
            return {"logits": self.lyrics_head(batch["lyrics_pool"])}
        la = self.audio_head(batch["audio_pool"])
        ll = self.lyrics_head(batch["lyrics_pool"])
        pa = F.softmax(la, dim=1)
        pl = F.softmax(ll, dim=1)
        p = self.alpha * pa + (1.0 - self.alpha) * pl
        return {
            "logits": torch.log(p.clamp_min(1e-8)),
            "aux_logits": [la, ll],
            "probs": p,
        }


class GatedMultimodalFusion(nn.Module):
    def __init__(self, audio_dim, text_dim, modality="bimodal", d_model=256,
                 hidden=128, dropout=0.3, gate_type="vector", n_classes=4):
        super().__init__()
        self.modality = modality
        self.gate_type = gate_type
        self.a_proj = nn.Linear(audio_dim, d_model) if modality in ("audio", "bimodal") else None
        self.l_proj = nn.Linear(text_dim, d_model) if modality in ("text", "bimodal") else None
        if modality == "bimodal":
            gate_out = d_model if gate_type == "vector" else 1
            self.gate = nn.Sequential(nn.Linear(2 * d_model, gate_out), nn.Sigmoid())
        else:
            self.gate = None
        self.head = MLP(d_model, [hidden], n_classes, dropout)

    def forward(self, batch):
        if self.modality == "audio":
            return {"logits": self.head(self.a_proj(batch["audio_pool"]))}
        if self.modality == "text":
            return {"logits": self.head(self.l_proj(batch["lyrics_pool"]))}
        ha = self.a_proj(batch["audio_pool"])
        hl = self.l_proj(batch["lyrics_pool"])
        g = self.gate(torch.cat([ha, hl], dim=1))
        h = g * ha + (1.0 - g) * hl
        return {"logits": self.head(h), "gate": g}


def build_model(model_name, modality, params=None):
    p = dict(params or {})
    dropout = float(p.get("dropout", 0.3))

    if model_name == MODEL_EARLY:
        return MERTRoBERTaEarlyFusion(
            AUDIO_DIM, TEXT_DIM, modality,
            hidden=int(p.get("hidden", 1024)),
            hidden2=int(p.get("hidden2", 256)),
            dropout=dropout,
        )
    if model_name == MODEL_MOODNET:
        return MoodNet3(
            AUDIO_DIM, TEXT_DIM, modality,
            width=int(p.get("width", 256)),
            hidden=int(p.get("hidden", 1024)),
            dropout=dropout,
        )
    if model_name == MODEL_CRNN:
        return CRNNBiGRULateFusion(
            AUDIO_DIM, TEXT_DIM, modality,
            cnn_dim=int(p.get("cnn_dim", 256)),
            gru_dim=int(p.get("gru_dim", 128)),
            dropout=dropout,
        )
    if model_name == MODEL_JOINT:
        d_model = int(p.get("d_model", 256))
        n_heads = int(p.get("n_heads", 4))
        if d_model % n_heads != 0:
            n_heads = 4
        return JointTransformer(
            AUDIO_DIM, TEXT_DIM, modality,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=int(p.get("n_layers", 2)),
            dropout=dropout,
        )
    if model_name == MODEL_WEIGHTED:
        return WeightedAverageFusion(
            AUDIO_DIM, TEXT_DIM, modality,
            hidden=int(p.get("hidden", 256)),
            dropout=dropout,
        )
    if model_name == MODEL_GATED:
        return GatedMultimodalFusion(
            AUDIO_DIM, TEXT_DIM, modality,
            d_model=int(p.get("d_model", 256)),
            hidden=int(p.get("hidden", 128)),
            dropout=dropout,
            gate_type=p.get("gate_type", "vector"),
        )
    raise ValueError(f"Model tidak dikenal: {model_name}")


# Smoke test bentuk tensor; tidak memerlukan dataset atau pretrained checkpoint.
def _smoke_batch(b=2):
    return {
        "audio_pool": torch.randn(b, AUDIO_DIM),
        "lyrics_pool": torch.randn(b, TEXT_DIM),
        "audio_seq": torch.randn(b, AUDIO_SEQ_LEN, AUDIO_DIM),
        "lyrics_seq": torch.randn(b, TEXT_SEQ_LEN, TEXT_DIM),
    }


if QUICK_RUN:
    print("Smoke test model (CPU):")
    for _name in MODEL_NAMES:
        for _mod in MODALITIES:
            _m = build_model(_name, _mod, {}).cpu().eval()
            with torch.no_grad():
                _out = _m(_smoke_batch())
            _logits = _out["logits"]
            assert _logits.shape == (2, 4), (_name, _mod, _logits.shape)
            _n = sum(p.numel() for p in _m.parameters()) / 1e6
            print(f"  {_name:42s} {_mod:8s} logits={tuple(_logits.shape)} params={_n:.2f}M")

## 17. Training, early stopping, dan Optuna

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def move_batch(batch, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


def forward_model(model, batch):
    out = model(batch)
    return out if isinstance(out, dict) else {"logits": out}


def loss_for(out, y, criterion, aux_weight=0.0):
    loss = criterion(out["logits"], y)
    if aux_weight > 0 and "aux_logits" in out:
        for aux in out["aux_logits"]:
            loss = loss + aux_weight * criterion(aux, y)
    return loss


@torch.no_grad()
def predict_loader(model, loader, device, criterion=None, collect_gate=False):
    model.eval()
    ys, ps, probs, gates = [], [], [], []
    aux = [[], []]
    total_loss, n_seen = 0.0, 0
    for batch in loader:
        batch = move_batch(batch, device)
        out = forward_model(model, batch)
        y = batch["label"]
        logits = out["logits"]
        if criterion is not None:
            total_loss += loss_for(out, y, criterion, 0.0).item() * len(y)
        n_seen += len(y)
        ys.append(y.cpu().numpy())
        ps.append(logits.argmax(dim=1).cpu().numpy())
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        if "aux_logits" in out:
            for j in range(min(2, len(out["aux_logits"]))):
                aux[j].append(torch.softmax(out["aux_logits"][j], dim=1).cpu().numpy())
        if collect_gate and "gate" in out:
            g = out["gate"]
            gates.append(g.mean(dim=-1).cpu().numpy())
    res = {
        "y_true": np.concatenate(ys),
        "y_pred": np.concatenate(ps),
        "probs": np.concatenate(probs),
        "loss": total_loss / max(n_seen, 1),
    }
    if aux[0]:
        res["aux_probs"] = [np.concatenate(aux[0]), np.concatenate(aux[1])]
    if gates:
        res["gate"] = np.concatenate(gates)
    return res


def make_optimizer(name, parameters, lr, weight_decay):
    if name == "sgd":
        return torch.optim.SGD(parameters, lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True)
    if name == "adam":
        return torch.optim.Adam(parameters, lr=lr, weight_decay=weight_decay)
    if name == "adamax":
        return torch.optim.Adamax(parameters, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Optimizer tidak dikenal: {name}")


def train_model(model_name, modality, params, split, seed=SEED,
                max_epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE,
                trial=None, verbose=False, eval_test=False):
    set_seed(seed)
    bs = int(params.get("batch_size", 32))
    # Untuk data kecil, jangan sampai hanya ada satu gradient step per epoch.
    bs = min(bs, max(2, len(split["train"]) // 4))
    train_loader = make_loader(split["train"], bs, shuffle=True)
    val_loader = make_loader(split["validation"], max(bs, 16), shuffle=False)
    test_loader = make_loader(split["test"], max(bs, 16), shuffle=False) if eval_test else None

    model = build_model(model_name, modality, params).to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_params = sum(p.numel() for p in trainable)

    class_weight = None
    if params.get("class_weight", "none") == "balanced":
        cnt = np.bincount(Y[split["train"]], minlength=len(CLASS_ORDER)).astype(np.float64)
        w = cnt.sum() / (len(CLASS_ORDER) * np.maximum(cnt, 1.0))
        class_weight = torch.tensor(w, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weight,
        label_smoothing=float(params.get("label_smoothing", 0.0)),
    )
    opt = make_optimizer(
        params.get("optimizer", "adam"), trainable,
        float(params.get("lr", 3e-4)), float(params.get("weight_decay", 1e-2)),
    )
    sched_name = params.get("scheduler", "plateau")
    if sched_name == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    elif sched_name == "plateau":
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="max", factor=0.5, patience=3, min_lr=1e-6
        )
    else:
        sched = None
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    aux_weight = float(params.get("aux_weight", 0.0)) if modality == "bimodal" else 0.0

    best_acc, best_f1 = -1.0, -1.0
    best_state = None
    bad = 0
    history = []
    t0 = time.time()

    for epoch in range(1, max_epochs + 1):
        model.train()
        tr_loss, n_seen = 0.0, 0
        tr_true, tr_pred = [], []
        for batch in train_loader:
            if len(batch["label"]) < 2:
                continue
            batch = move_batch(batch, device)
            opt.zero_grad(set_to_none=True)
            amp_ctx = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
            with amp_ctx:
                out = forward_model(model, batch)
                loss = loss_for(out, batch["label"], criterion, aux_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt)
            scaler.update()
            tr_loss += loss.item() * len(batch["label"])
            n_seen += len(batch["label"])
            tr_true.append(batch["label"].detach().cpu().numpy())
            tr_pred.append(out["logits"].argmax(dim=1).detach().cpu().numpy())

        train_loss = tr_loss / max(n_seen, 1)
        train_acc = accuracy_score(np.concatenate(tr_true), np.concatenate(tr_pred)) if n_seen else 0.0
        val = predict_loader(model, val_loader, device, criterion)
        val_metrics = compute_metrics(val["y_true"], val["y_pred"])

        if sched is not None:
            if sched_name == "plateau":
                sched.step(val_metrics["accuracy"])
            else:
                sched.step()

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val["loss"],
            "train_accuracy": train_acc,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision_macro": val_metrics["precision_macro"],
            "val_recall_macro": val_metrics["recall_macro"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "lr": opt.param_groups[0]["lr"],
        }
        history.append(row)
        if verbose:
            print(f"  epoch {epoch:02d} loss={train_loss:.4f} val_acc={row['val_accuracy']:.4f} "
                  f"val_f1={row['val_macro_f1']:.4f}")

        if trial is not None:
            trial.report(val_metrics["macro_f1"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        # Early stopping dan checkpoint berdasarkan validation accuracy (sesuai ketentuan).
        if val_metrics["accuracy"] > best_acc + 1e-6:
            best_acc = val_metrics["accuracy"]
            best_f1 = val_metrics["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    res = {
        "model": model,
        "history": history,
        "n_params": n_params,
        "best_val_accuracy": best_acc,
        "best_val_macro_f1": best_f1,
        "epochs_run": len(history),
        "train_time_sec": time.time() - t0,
        "val": predict_loader(model, val_loader, device, criterion),
    }
    if eval_test:
        test = predict_loader(model, test_loader, device, criterion)
        res["test"] = test
        res["test_metrics"] = compute_metrics(test["y_true"], test["y_pred"])
    return res


def tune_alpha(aux_probs, y_val):
    best_a, best_score = 0.5, -1.0
    for a in np.linspace(0.0, 1.0, 101):
        p = a * aux_probs[0] + (1.0 - a) * aux_probs[1]
        score = f1_score(y_val, p.argmax(axis=1), average="macro", zero_division=0)
        if score > best_score:
            best_a, best_score = float(a), float(score)
    return best_a, best_score


def suggest_params(trial, model_name, modality):
    p = {
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamax", "sgd"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "batch_size": trial.suggest_categorical(
            "batch_size", BATCH_CHOICES_AUDIO if modality == "audio" else BATCH_CHOICES_BIMODAL
        ),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "label_smoothing": trial.suggest_categorical("label_smoothing", [0.0, 0.05, 0.1]),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced"]),
        "scheduler": trial.suggest_categorical("scheduler", ["plateau", "cosine", "none"]),
    }
    if model_name == MODEL_EARLY:
        p["hidden"] = trial.suggest_categorical("hidden", [512, 1024, 2048])
        p["hidden2"] = trial.suggest_categorical("hidden2", [128, 256, 512])
    elif model_name == MODEL_MOODNET:
        p["width"] = trial.suggest_categorical("width", [128, 256])
        p["hidden"] = trial.suggest_categorical("hidden", [512, 1024])
    elif model_name == MODEL_CRNN:
        p["cnn_dim"] = trial.suggest_categorical("cnn_dim", [128, 256])
        p["gru_dim"] = trial.suggest_categorical("gru_dim", [64, 128, 256])
        p["aux_weight"] = trial.suggest_categorical("aux_weight", [0.0, 0.25, 0.5])
    elif model_name == MODEL_JOINT:
        p["d_model"] = trial.suggest_categorical("d_model", [128, 256])
        p["n_heads"] = trial.suggest_categorical("n_heads", [4, 8])
        p["n_layers"] = trial.suggest_categorical("n_layers", [1, 2, 3])
    elif model_name == MODEL_WEIGHTED:
        p["hidden"] = trial.suggest_categorical("hidden", [128, 256, 512])
        p["aux_weight"] = trial.suggest_categorical("aux_weight", [0.25, 0.5])
    elif model_name == MODEL_GATED:
        p["d_model"] = trial.suggest_categorical("d_model", [128, 256])
        p["hidden"] = trial.suggest_categorical("hidden", [128, 256])
        p["gate_type"] = trial.suggest_categorical("gate_type", ["vector", "scalar"])
    return p


def best_params_path(scenario, rep, model_name, modality):
    folder = OPTUNA_DIR / scenario / f"rep{rep}"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{variant_slug(model_name, modality)}.json"


def run_optuna(scenario, rep, model_name, modality, n_trials):
    split = SPLITS[(scenario, rep)]
    path = best_params_path(scenario, rep, model_name, modality)
    if RESUME and path.exists():
        return json.loads(path.read_text())

    study_path = OPTUNA_DIR / scenario / f"rep{rep}" / f"{variant_slug(model_name, modality)}.db"
    study = optuna.create_study(
        study_name=f"{variant_slug(model_name, modality)}_{scenario}_rep{rep}",
        direction="maximize",
        storage=f"sqlite:///{study_path.as_posix()}",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=TUNE_SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=2),
    )

    def objective(trial):
        params = suggest_params(trial, model_name, modality)
        try:
            res = train_model(
                model_name, modality, params, split, seed=TUNE_SEED,
                max_epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE,
                trial=trial, verbose=False, eval_test=False,
            )
            return float(res["best_val_macro_f1"])
        except optuna.TrialPruned:
            raise
        except RuntimeError as exc:
            print(f"  [trial {trial.number}] gagal: {exc}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return 0.0

    t0 = time.time()
    remaining = max(0, int(n_trials) - len(study.trials))
    if remaining > 0:
        study.optimize(objective, n_trials=remaining)
    if len(study.trials) == 0:
        raise RuntimeError(f"Tidak ada trial Optuna untuk {model_name}/{modality}")
    payload = {
        "scenario": scenario,
        "repeat": rep,
        "model": model_name,
        "modality": modality,
        "best_value": float(study.best_value),
        "best_params": study.best_params,
        "n_trials": len(study.trials),
        "duration_sec": time.time() - t0,
    }
    path.write_text(json.dumps(payload, indent=2, default=float))
    pd.DataFrame([{
        "number": t.number, "state": str(t.state), "value": t.value,
        **{f"param_{k}": v for k, v in t.params.items()},
    } for t in study.trials]).to_csv(path.with_suffix(".trials.csv"), index=False)
    print(f"  best {variant_label(model_name, modality)}: {study.best_value:.4f} "
          f"({len(study.trials)} trials, {payload['duration_sec']:.0f}s)")
    return payload

## 18. Menjalankan hyperparameter tuning

Tuning bimodal dilakukan penuh. Untuk `audio` dan `text`, secara default hyperparameter
bimodal dipakai kembali agar perbandingan modalitas tidak menghabiskan anggaran tuning
tiga kali lipat; set `TUNE_UNIMODAL=True` untuk tuning terpisah. Jika tetap `False`,
varian unimodal adalah eksperimen turunan, bukan hasil tuning independen.

In [ ]:
TUNE_UNIMODAL = False
best_params_all = {}
tune_t0 = time.time()
for scenario in SCENARIOS:
    for rep in range(1, REPEATS + 1):
        for model_name in MODEL_NAMES:
            payload = run_optuna(scenario, rep, model_name, "bimodal", N_TRIALS)
            best_params_all[(scenario, rep, model_name, "bimodal")] = payload["best_params"]
            for modality in ["audio", "text"]:
                if TUNE_UNIMODAL:
                    payload_u = run_optuna(scenario, rep, model_name, modality, N_TRIALS_UNIMODAL)
                    best_params_all[(scenario, rep, model_name, modality)] = payload_u["best_params"]
                else:
                    inherited = dict(payload["best_params"])
                    inherited["aux_weight"] = 0.0
                    if modality == "audio":
                        inherited["batch_size"] = max(32, int(inherited.get("batch_size", 32)))
                    path_u = best_params_path(scenario, rep, model_name, modality)
                    if not path_u.exists() or not RESUME:
                        path_u.write_text(json.dumps({
                            "scenario": scenario, "repeat": rep, "model": model_name,
                            "modality": modality, "best_value": None,
                            "best_params": inherited,
                            "inherited_from": "bimodal",
                            "n_trials": 0, "duration_sec": 0.0,
                        }, indent=2, default=float))
                    best_params_all[(scenario, rep, model_name, modality)] = inherited
print(f"Tuning selesai dalam {(time.time() - tune_t0) / 60:.1f} menit.")

# Simpan tabel hyperparameter terbaik
rows = []
for scenario in SCENARIOS:
    for rep in range(1, REPEATS + 1):
        for model_name, modality in MODEL_VARIANTS:
            p = best_params_path(scenario, rep, model_name, modality)
            if p.exists():
                payload = json.loads(p.read_text())
                rows.append({
                    "scenario": scenario, "repeat": rep, "model": model_name,
                    "modality": modality,
                    "best_value": payload.get("best_value"),
                    "inherited_from": payload.get("inherited_from", ""),
                    "n_trials": payload.get("n_trials", 0),
                    "params_json": json.dumps(payload.get("best_params", {})),
                })
pd.DataFrame(rows).to_csv(EXP / "best_hyperparameters.csv", index=False)
print("Hyperparameter terbaik ->", EXP / "best_hyperparameters.csv")

## 19. Final training dan evaluasi test

Setiap kombinasi model × modalitas × skenario × repeat dilatih dengan konfigurasi terbaik.
Test hanya dipakai sekali di akhir. Untuk Weighted Average bimodal, α dipilih dari validation.

In [ ]:
results_path = EXP / "evaluation_results.csv"
history_path = EXP / "training_history.csv"
done_keys = set()
if results_path.exists():
    _old = pd.read_csv(results_path)
    done_keys = set(zip(_old.scenario, _old.repeat, _old.model, _old.modality, _old.seed.astype(int)))
    all_results = _old.to_dict("records")
else:
    all_results = []
if history_path.exists():
    all_history = pd.read_csv(history_path).to_dict("records")
else:
    all_history = []
for scenario in SCENARIOS:
    for rep in range(1, REPEATS + 1):
        split = SPLITS[(scenario, rep)]
        for model_name, modality in MODEL_VARIANTS:
            path = best_params_path(scenario, rep, model_name, modality)
            if not path.exists():
                print("SKIP, params tidak ada:", path)
                continue
            payload = json.loads(path.read_text())
            params = payload["best_params"]
            for seed in FINAL_SEEDS:
                key = (scenario, rep, model_name, modality, seed)
                if RESUME and key in done_keys:
                    continue
                print(f"Train {variant_label(model_name, modality)} | {scenario} rep{rep} seed{seed}")
                res = train_model(
                    model_name, modality, params, split, seed=seed,
                    max_epochs=MAX_EPOCHS, patience=PATIENCE, device=DEVICE,
                    verbose=False, eval_test=True,
                )
                model = res["model"]
                alpha = np.nan
                if model_name == MODEL_WEIGHTED and modality == "bimodal":
                    alpha, alpha_score = tune_alpha(res["val"]["aux_probs"], res["val"]["y_true"])
                    model.set_alpha(alpha)
                    test_loader = make_loader(split["test"], max(int(params.get("batch_size", 32)), 16), False)
                    test = predict_loader(model, test_loader, DEVICE, collect_gate=True)
                    res["test"] = test
                    res["test_metrics"] = compute_metrics(test["y_true"], test["y_pred"])
                test = res["test"]
                tm = res["test_metrics"]

                slug = variant_slug(model_name, modality)
                out_dir = EXP / scenario / slug
                ckpt_dir = CKPT / scenario / slug
                out_dir.mkdir(parents=True, exist_ok=True)
                ckpt_dir.mkdir(parents=True, exist_ok=True)
                run_tag = f"rep{rep}_seed{seed}"
                np.savez(
                    out_dir / f"{run_tag}.npz",
                    y_true=test["y_true"], y_pred=test["y_pred"], probs=test["probs"],
                    aux0=test.get("aux_probs", [np.array([]), np.array([])])[0],
                    aux1=test.get("aux_probs", [np.array([]), np.array([])])[1],
                    gate=test.get("gate", np.array([])),
                    alpha=np.array([alpha]),
                )
                torch.save(model.state_dict(), ckpt_dir / f"{run_tag}.pt")
                pd.DataFrame(res["history"]).to_csv(out_dir / f"{run_tag}_history.csv", index=False)

                row = {
                    "scenario": scenario, "repeat": rep, "model": model_name,
                    "modality": modality, "experiment": variant_label(model_name, modality),
                    "seed": seed, **tm,
                    "val_accuracy": res["best_val_accuracy"],
                    "val_macro_f1": res["best_val_macro_f1"],
                    "best_epoch": int(np.argmax([h["val_accuracy"] for h in res["history"]]) + 1),
                    "epochs_run": res["epochs_run"],
                    "n_train": len(split["train"]), "n_val": len(split["validation"]),
                    "n_test": len(split["test"]), "n_params": res["n_params"],
                    "train_time_sec": res["train_time_sec"], "alpha": alpha,
                    "fusion_weight": (
                        float(torch.sigmoid(model.fusion_logit).detach().cpu().item())
                        if model_name == MODEL_CRNN and modality == "bimodal" else np.nan
                    ),
                }
                all_results.append(row)
                for h in res["history"]:
                    all_history.append({
                        "scenario": scenario, "repeat": rep, "model": model_name,
                        "modality": modality, "seed": seed, **h,
                    })
                pd.DataFrame(all_results).to_csv(results_path, index=False)
                pd.DataFrame(all_history).to_csv(history_path, index=False)
                del model, res
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

print("Evaluasi selesai.")
if results_path.exists():
    print(pd.read_csv(results_path).tail().to_string(index=False))

## 20. Perbandingan skenario dan tabel evaluasi

In [ ]:
if not results_path.exists():
    raise FileNotFoundError("evaluation_results.csv belum ada; jalankan final training terlebih dahulu.")

res_df = pd.read_csv(results_path)
METRICS = ["accuracy", "precision_macro", "recall_macro", "macro_f1", "weighted_f1"]
LABELS = {
    "accuracy": "Accuracy", "precision_macro": "Precision Macro",
    "recall_macro": "Recall Macro", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1",
}
summary_rows = []
for (scenario, experiment), g in res_df.groupby(["scenario", "experiment"], sort=False):
    row = {"scenario": scenario, "Experiment": experiment, "n_runs": len(g)}
    for m in METRICS:
        row[f"{LABELS[m]} Mean"] = g[m].mean()
        row[f"{LABELS[m]} Std"] = g[m].std(ddof=1)
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(EXP / "model_comparison.csv", index=False)

for scenario in SCENARIOS:
    table = summary_df[summary_df.scenario == scenario].drop(columns=["scenario"]).copy()
    table = table[[
        "Experiment",
        "Accuracy Mean", "Accuracy Std", "Precision Macro Mean", "Precision Macro Std",
        "Recall Macro Mean", "Recall Macro Std", "Macro F1 Mean", "Macro F1 Std",
        "Weighted F1 Mean", "Weighted F1 Std",
    ]].sort_values("Macro F1 Mean", ascending=False)
    table.round(4).to_csv(EXP / f"model_comparison_{scenario}.csv", index=False)
    print(f"\n===== {scenario} =====")
    print(table.round(4).to_string(index=False))
print("\nTabel perbandingan ->", EXP / "model_comparison.csv")
print("Catatan: mean/std dihitung dari repeated holdout/training seed yang tercantum di evaluation_results.csv.")

## 21. Confusion matrix, classification report, dan visualisasi

In [ ]:
for scenario in SCENARIOS:
    variants = sorted(res_df.loc[res_df.scenario == scenario, "experiment"].unique())
    ncols = 3
    nrows = int(math.ceil(len(variants) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.5 * nrows))
    axes = np.asarray(axes).reshape(-1)
    for ax, experiment in zip(axes, variants):
        g = res_df[(res_df.scenario == scenario) & (res_df.experiment == experiment)]
        cm_total = np.zeros((4, 4), dtype=int)
        for _, r in g.iterrows():
            slug = variant_slug(r["model"], r["modality"])
            p = EXP / scenario / slug / f"rep{int(r['repeat'])}_seed{int(r['seed'])}.npz"
            z = np.load(p)
            cm_total += confusion_matrix(z["y_true"], z["y_pred"], labels=list(range(4)))
        cmn = cm_total / np.maximum(cm_total.sum(1, keepdims=True), 1)
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(4)); ax.set_yticks(range(4))
        ax.set_xticklabels(CLASS_ORDER); ax.set_yticklabels(CLASS_ORDER)
        ax.set_title(experiment, fontsize=8)
        for i in range(4):
            for j in range(4):
                ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm_total[i, j]})",
                        ha="center", va="center", fontsize=6,
                        color="white" if cmn[i, j] > 0.5 else "black")
    for ax in axes[len(variants):]:
        ax.axis("off")
    fig.suptitle(f"Confusion matrix (akumulasi repeat), {scenario}")
    fig.tight_layout()
    fig.savefig(FIG / f"confusion_{scenario}.png", dpi=160)
    plt.show()

# Classification report per run
report_rows = []
for _, r in res_df.iterrows():
    slug = variant_slug(r["model"], r["modality"])
    z = np.load(EXP / r["scenario"] / slug / f"rep{int(r['repeat'])}_seed{int(r['seed'])}.npz")
    rep = classification_report(
        z["y_true"], z["y_pred"], labels=list(range(4)), target_names=CLASS_ORDER,
        output_dict=True, zero_division=0,
    )
    for label in CLASS_ORDER + ["macro avg", "weighted avg"]:
        if label in rep:
            report_rows.append({
                "scenario": r["scenario"], "repeat": r["repeat"], "model": r["model"],
                "modality": r["modality"], "seed": r["seed"], "class": label,
                **{k: rep[label][k] for k in ["precision", "recall", "f1-score", "support"]},
            })
pd.DataFrame(report_rows).to_csv(EXP / "classification_report.csv", index=False)

# Confusion matrix dalam format long CSV
cm_rows = []
for _, r in res_df.iterrows():
    slug = variant_slug(r["model"], r["modality"])
    z = np.load(EXP / r["scenario"] / slug / f"rep{int(r['repeat'])}_seed{int(r['seed'])}.npz")
    cm = confusion_matrix(z["y_true"], z["y_pred"], labels=list(range(len(CLASS_ORDER))))
    for i, true_label in enumerate(CLASS_ORDER):
        for j, pred_label in enumerate(CLASS_ORDER):
            cm_rows.append({
                "scenario": r["scenario"], "repeat": r["repeat"], "model": r["model"],
                "modality": r["modality"], "seed": r["seed"],
                "true_label": true_label, "pred_label": pred_label, "count": int(cm[i, j]),
            })
pd.DataFrame(cm_rows).to_csv(EXP / "confusion_matrix.csv", index=False)

# Grafik perbandingan metrik
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, scenario in zip(axes, SCENARIOS):
    table = summary_df[summary_df.scenario == scenario].sort_values("Macro F1 Mean")
    ax.barh(range(len(table)), table["Macro F1 Mean"], xerr=table["Macro F1 Std"],
            color="#4C78A8", capsize=3)
    ax.set_yticks(range(len(table)))
    ax.set_yticklabels(table.Experiment, fontsize=7)
    ax.set_xlabel("Macro F1")
    ax.set_title(scenario)
    ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "comparison_macro_f1.png", dpi=160)
plt.show()

# Heatmap model x skenario
pivot = summary_df.pivot(index="Experiment", columns="scenario", values="Macro F1 Mean")
fig, ax = plt.subplots(figsize=(7, 8))
im = ax.imshow(pivot.values, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns, fontsize=8)
ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index, fontsize=7)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
fig.tight_layout()
fig.savefig(FIG / "heatmap_macro_f1.png", dpi=160)
plt.show()

# Kurva training: loss dan accuracy train/validation, per model dan skenario (bimodal).
# Histori untuk modalitas audio/text tetap tersimpan di training_history.csv.
if history_path.exists():
    hist = pd.read_csv(history_path)
    for model_name in MODEL_NAMES:
        h = hist[(hist.model == model_name) & (hist.modality == "bimodal")]
        if h.empty:
            continue
        fig, axes = plt.subplots(2, 2, figsize=(10, 7))
        for row, scenario in enumerate(SCENARIOS):
            hs = h[h.scenario == scenario]
            for col, (tr_col, va_col, title) in enumerate([
                ("train_loss", "val_loss", "Loss"),
                ("train_accuracy", "val_accuracy", "Accuracy"),
            ]):
                ax = axes[row, col]
                if not hs.empty:
                    for col_name, lab, color in [(tr_col, "train", "#4C78A8"),
                                                 (va_col, "validation", "#F58518")]:
                        g = hs.groupby("epoch")[col_name].agg(["mean", "std"])
                        ax.plot(g.index, g["mean"], label=lab, color=color)
                        ax.fill_between(g.index,
                                        g["mean"] - g["std"].fillna(0),
                                        g["mean"] + g["std"].fillna(0),
                                        color=color, alpha=0.18)
                ax.set_title(f"{scenario} - {title}")
                ax.set_xlabel("Epoch")
                ax.grid(alpha=0.3)
                ax.legend()
        fig.suptitle(f"{model_name} (bimodal)")
        fig.tight_layout()
        fig.savefig(FIG / f"training_{slugify(model_name)}.png", dpi=160)
        plt.show()

## 22. Analisis fusion dan kontribusi modalitas

In [ ]:
fusion_rows = []
for scenario in SCENARIOS:
    for model_name in MODEL_NAMES:
        g = res_df[(res_df.scenario == scenario) & (res_df.model == model_name)]
        means = g.groupby("modality")["macro_f1"].mean()
        if set(["audio", "text", "bimodal"]).issubset(means.index):
            fusion_rows.append({
                "scenario": scenario, "model": model_name,
                "audio_only": means["audio"], "text_only": means["text"],
                "bimodal": means["bimodal"],
                "gain_over_audio": means["bimodal"] - means["audio"],
                "gain_over_text": means["bimodal"] - means["text"],
            })
fusion_df = pd.DataFrame(fusion_rows)
fusion_df.to_csv(EXP / "fusion_analysis.csv", index=False)
print("Fusion analysis:")
print(fusion_df.round(4).to_string(index=False))

# Gate analysis untuk Gated Multimodal Fusion
gate_rows = []
for scenario in SCENARIOS:
    for rep in range(1, REPEATS + 1):
        path = best_params_path(scenario, rep, MODEL_GATED, "bimodal")
        if not path.exists():
            continue
        params = json.loads(path.read_text())["best_params"]
        for seed in FINAL_SEEDS:
            ckpt_path = CKPT / scenario / variant_slug(MODEL_GATED, "bimodal") / f"rep{rep}_seed{seed}.pt"
            if not ckpt_path.exists():
                continue
            model = build_model(MODEL_GATED, "bimodal", params).to(DEVICE)
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
            model.eval()
            loader = make_loader(SPLITS[(scenario, rep)]["test"], 32, False)
            out = predict_loader(model, loader, DEVICE, collect_gate=True)
            for c, label in enumerate(CLASS_ORDER):
                mask = out["y_true"] == c
                if mask.any():
                    gate_rows.append({
                        "scenario": scenario, "repeat": rep, "seed": seed,
                        "class": label, "gate_audio_mean": float(out["gate"][mask].mean()),
                        "n": int(mask.sum()),
                    })
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
pd.DataFrame(gate_rows).to_csv(EXP / "gate_analysis.csv", index=False)
if gate_rows:
    gate_df = pd.DataFrame(gate_rows)
    print("\nRata-rata gate audio per kelas:")
    print(gate_df.groupby(["scenario", "class"]).gate_audio_mean.mean().round(4).to_string())

# Alpha analysis untuk Weighted Average
if "alpha" in res_df.columns:
    alpha_df = res_df[(res_df.model == MODEL_WEIGHTED) & (res_df.modality == "bimodal") &
                      res_df.alpha.notna()][["scenario", "repeat", "seed", "alpha"]]
    alpha_df.to_csv(EXP / "alpha_analysis.csv", index=False)
    if len(alpha_df):
        print("\nAlpha Weighted Average:")
        print(alpha_df.groupby("scenario").alpha.agg(["mean", "std"]).round(4).to_string())

# Occlusion modalitas untuk model bimodal: zero-kan audio atau lirik saat inference.
@torch.no_grad()
def predict_with_drop(model, loader, device, drop=None):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        batch = move_batch(batch, device)
        if drop == "audio":
            batch["audio_pool"] = torch.zeros_like(batch["audio_pool"])
            batch["audio_seq"] = torch.zeros_like(batch["audio_seq"])
        elif drop == "text":
            batch["lyrics_pool"] = torch.zeros_like(batch["lyrics_pool"])
            batch["lyrics_seq"] = torch.zeros_like(batch["lyrics_seq"])
        out = forward_model(model, batch)
        ys.append(batch["label"].detach().cpu().numpy())
        ps.append(out["logits"].argmax(dim=1).detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


occ_rows = []
for scenario in SCENARIOS:
    for rep in range(1, REPEATS + 1):
        for model_name in MODEL_NAMES:
            path = best_params_path(scenario, rep, model_name, "bimodal")
            if not path.exists():
                continue
            params = json.loads(path.read_text())["best_params"]
            for seed in FINAL_SEEDS:
                ckpt_path = CKPT / scenario / variant_slug(model_name, "bimodal") / f"rep{rep}_seed{seed}.pt"
                if not ckpt_path.exists():
                    continue
                model = build_model(model_name, "bimodal", params).to(DEVICE)
                model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
                model.eval()
                loader = make_loader(SPLITS[(scenario, rep)]["test"], 32, False)
                y_full, p_full = predict_with_drop(model, loader, DEVICE, None)
                y_a, p_a = predict_with_drop(model, loader, DEVICE, "audio")
                y_t, p_t = predict_with_drop(model, loader, DEVICE, "text")
                f_full = f1_score(y_full, p_full, average="macro", zero_division=0)
                f_a = f1_score(y_a, p_a, average="macro", zero_division=0)
                f_t = f1_score(y_t, p_t, average="macro", zero_division=0)
                occ_rows.append({
                    "scenario": scenario, "repeat": rep, "model": model_name,
                    "seed": seed, "full_macro_f1": f_full,
                    "drop_audio_macro_f1": f_a, "drop_text_macro_f1": f_t,
                    "delta_audio": f_full - f_a, "delta_text": f_full - f_t,
                })
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
pd.DataFrame(occ_rows).to_csv(EXP / "modality_occlusion.csv", index=False)
if occ_rows:
    print("\nOcclusion modalitas (bimodal):")
    print(pd.DataFrame(occ_rows).groupby(["scenario", "model"])[
        ["full_macro_f1", "drop_audio_macro_f1", "drop_text_macro_f1", "delta_audio", "delta_text"]
    ].mean().round(4).to_string())

## 23. Rekomendasi model, kesimpulan, dan status

In [ ]:
agg = res_df.groupby(["scenario", "experiment", "model", "modality"]).agg(
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    accuracy_mean=("accuracy", "mean"),
    weighted_f1_mean=("weighted_f1", "mean"),
    n_params=("n_params", "mean"),
    train_time_sec=("train_time_sec", "mean"),
    n_runs=("macro_f1", "count"),
).reset_index()
agg["rank_in_scenario"] = agg.groupby("scenario")["macro_f1_mean"].rank(ascending=False, method="min")
agg.to_csv(EXP / "recommendation_table.csv", index=False)

overall = res_df.groupby(["experiment", "model", "modality"]).agg(
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    accuracy_mean=("accuracy", "mean"),
    weighted_f1_mean=("weighted_f1", "mean"),
    n_params=("n_params", "mean"),
    train_time_sec=("train_time_sec", "mean"),
    n_runs=("macro_f1", "count"),
).reset_index().sort_values("macro_f1_mean", ascending=False)
overall.to_csv(EXP / "recommendation_overall.csv", index=False)

print("Peringkat per skenario:")
print(agg.sort_values(["scenario", "rank_in_scenario"]).round(4).to_string(index=False))
print("\nRingkasan gabungan kedua skenario:")
print(overall.round(4).to_string(index=False))

if len(overall):
    best = overall.iloc[0]
    print(f"\nModel dengan Macro F1 rata-rata tertinggi: {best['experiment']}")
    print(f"  Macro F1 = {best['macro_f1_mean']:.4f} +/- {best['macro_f1_std']:.4f} "
          f"(n={int(best['n_runs'])} run), Accuracy = {best['accuracy_mean']:.4f}")
    best_runs = agg[agg.experiment == best["experiment"]]
    print("  Peringkat per skenario: " + ", ".join(
        f"{r['scenario']}=#{int(r['rank_in_scenario'])}" for _, r in best_runs.iterrows()
    ))
    print("\nRekomendasi harus mempertimbangkan std, jumlah parameter, waktu training, "
          "dan konsistensi antar-skenario. Selisih kecil belum tentu bermakna.")
    simple = overall[overall.modality == "bimodal"].sort_values("macro_f1_mean", ascending=False)
    if len(simple) > 1:
        top = simple.iloc[0]
        for _, row in simple.iloc[1:].iterrows():
            if row["macro_f1_mean"] >= top["macro_f1_mean"] - 0.01 and row["n_params"] < top["n_params"]:
                print(f"Alternatif lebih sederhana: {row['experiment']} "
                      f"(selisih Macro F1 {top['macro_f1_mean'] - row['macro_f1_mean']:+.4f}, "
                      f"parameter {row['n_params']/1e6:.2f}M vs {top['n_params']/1e6:.2f}M).")

print("\nFile hasil:")
for p in sorted(EXP.glob("*.csv")):
    print(" -", p.name)

## Petunjuk menjalankan dan status pengujian

1. Pastikan dataset berada di `MERGE_Bimodal_Complete/` atau set `MERGE_ROOT`.
2. Pasang dependensi: `pip install transformers optuna librosa soundfile scikit-learn pandas matplotlib`.
   PyTorch GPU harus dipasang sesuai versi CUDA mesin.
3. Untuk smoke test cepat:
   - PowerShell: `$env:MERGE_QUICK="1"`
   - cmd.exe: `set MERGE_QUICK=1`
   Lalu jalankan seluruh notebook. Mode ini memakai subset kecil, 5 detik audio,
   1 trial, dan 3 epoch; hasilnya bukan hasil penelitian dan disimpan di folder `*_quick`.
4. Untuk eksperimen penuh, set `MERGE_QUICK=0` dan jalankan berurutan.
   Ekstraksi fitur MERT pada CPU sangat lambat; GPU sangat disarankan.
5. Output utama ada di `experiments_mert/`, `checkpoints_mert/`, dan `optuna_mert/`.

**Status:** full run 2.172 lagu selesai pada 2026-11-18 dengan GPU RTX 3050 4 GB:
ekstraksi MERT 816 detik, RoBERTa 221 detik, 36 studi Optuna (5 trial each),
108 final run (18 varian x 2 skenario x 3 repeat), confusion matrix, classification report,
fusion analysis, occlusion, dan rekomendasi. Hasil ada di `experiments_mert/`.
Smoke test 24 lagu ada di `experiments_mert_quick/` dan bukan hasil penelitian.